In [ ]:
import torch
from transformers import AutoTokenizer,AutoModelForSeq2SeqLM
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnableLambda
from langchain_core.output_parsers import StrOutputParser
model_name="google/flan-t5-small"
tokenizer=AutoTokenizer.from_pretrained(model_name)
model=AutoModelForSeq2SeqLM.from_pretrained(model_name)
def local_llm(prompt):
    if hasattr(prompt,"to_string"):
        prompt=prompt.to_string()

    inputs=tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True
    )
    with torch.no_grad():
        output_ids=model.generate(
            **inputs,
            max_new_tokens=64
        )
    answer=tokenizer.decode(output_ids[0],skip_special_tokens=True)
    return answer
prompt_template=PromptTemplate.from_template(
    """
You are a helpful fitness assistant.
User Qusetion:
{question}
Answer in one clear sentence:
"""
)
llm_runnable=RunnableLambda(local_llm)
output_parser=StrOutputParser()
chain=prompt_template|llm_runnable|output_parser
result=chain.invoke({
    "question":"What should I eat during fat loss?"
})
print(result)

c:\Users\marce\ai-career-2026\ai_env\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 190/190 [00:00<00:00, 10516.06it/s]
The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


ValueError: text input must be of type `str` (single example), `list[str]` (batch or single pretokenized example) or `list[list[str]]` (batch of pretokenized examples) or `list[tuple[list[str], list[str]]]` (batch of pretokenized sequence pairs).